# Week 1 — Foundations: Cleaning & Exploring the Applicant Dataset
**Name:** Natasha Fatima

**Task:** Clean and explore a messy `applicants.csv` dataset using Pandas, and document every decision.

This notebook follows the NextGenLearners Week 1 task guide step by step: setup, data generation, exploration, missing-value handling, deduplication, text standardization, type fixing, a data quality summary, and export.

In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [4]:
df = pd.read_csv('applicants.csv')
df.head(10)

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,David Jenkins,georgemiller@example.com,0393-7923747,GRAPHIC DESIGN,NaN,16/10/2025,under review
1,Martin Ross,sheila14@example.org,0394-0196556,Graphic Design,LUMS,2026-02-21,REJECTED
2,Dana Terry,melissa91@example.net,0305-8651850,APP DEV,Comsats University,21/05/2026,under review
3,Carmen Rose,ibrandt@example.net,0334-6578713,digital marketing,Punjab University,10-05-2025,NaN
4,Eric Ortiz,jterry@example.org,0336-6299468,webdev,Punjab University,not_provided,UNDER REVIEW
5,Jennifer Brown,dcarlson@example.net,0348-0184514,APP DEV,University of the Punjab,2025-12-21,Rejected
6,Pamela Roberts,smithjulie@example.org,0326-8388516,GRAPHIC DESIGN,UET Lahore,26/05/2026,REJECTED
7,Ashley Delacruz,alexistyler@example.org,0395-0240268,graphic design,FAST NUCES,2025-08-25,Rejected
8,Michael Farrell,russellbeasley@example.com,0357-6615654,App Development,Punjab University,2025-11-28,selected
9,John Torres,gbender@example.net,0377-1906594,data science,Punjab University,24/07/2025,REJECTED


## Step 3: First Look — Understand Before Cleaning

Before touching anything, I looked at the shape, structure, and missing-value counts of the raw data.

In [5]:
df.shape

(95, 7)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Applicant Name    92 non-null     object
 1   Email             90 non-null     object
 2   Phone             95 non-null     object
 3   Domain Applied    95 non-null     object
 4   University        80 non-null     object
 5   Application Date  95 non-null     object
 6   Status            86 non-null     object
dtypes: object(7)
memory usage: 5.3+ KB


In [7]:
df.isnull().sum()

Applicant Name       3
Email                5
Phone                0
Domain Applied       0
University          15
Application Date     0
Status               9
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(3)

**Observation:** The raw dataset has **95 rows and 7 columns**. `pandas.info()` doesn't catch missing values that were saved as empty strings (`""`) rather than true `NaN` in the CSV, so before trusting the null counts I converted blank/whitespace-only strings to real `NaN` values in the key columns.

In [9]:
for col in ['Applicant Name', 'Email', 'University', 'Status']:
    df[col] = df[col].replace(r'^\s*$', pd.NA, regex=True)

df.isnull().sum()

Applicant Name       3
Email                5
Phone                0
Domain Applied       0
University          15
Application Date     0
Status               9
dtype: int64

**After that fix**, the real picture is: several missing `Applicant Name`, `Email`, `University`, and `Status` values, plus **3 exact duplicate rows** confirmed by `.duplicated().sum()`. `Application Date` is stored as `object` (text), not a real date .

## Step 4: Handle Missing Values (Column by Column, With Justification)

There is no single rule for missing data — each column gets its own decision, based on how important that field is:

- **Applicant Name missing → drop the row.** A nameless application can't be used for outreach or identification, so it isn't useful to keep.
- **Email missing → drop the row.** Email is the main way to follow up with an applicant, and I'm also using it as the key for deduplication, so rows without it aren't usable either.
- **University missing → fill with `"Not Specified"`.** University isn't essential for deciding an applicant's status, so there's no reason to lose the whole row over it.
- **Status missing → fill with `"Under Review"`.** A blank status most likely means the application hasn't been processed yet, so this is a sensible default rather than a guess.

In [10]:
before = len(df)
df = df.dropna(subset=['Applicant Name'])
print(f"Dropped {before - len(df)} rows with missing Applicant Name")

Dropped 3 rows with missing Applicant Name


In [11]:
before = len(df)
df = df.dropna(subset=['Email'])
print(f"Dropped {before - len(df)} rows with missing Email")

Dropped 3 rows with missing Email


In [12]:
df['University'] = df['University'].fillna('Not Specified')
df['Status'] = df['Status'].fillna('Under Review')

df.isnull().sum()

Applicant Name      0
Email               0
Phone               0
Domain Applied      0
University          0
Application Date    0
Status              0
dtype: int64

All missing values in critical fields are now resolved: no more nulls anywhere in the dataframe.

## Step 5: Remove Duplicate Applicants

First I removed **exact duplicate rows** (identical across every column). Then, since `drop_duplicates()` alone won't catch someone who applied twice with slightly different name formatting, I checked for **near-duplicates by a cleaned version of `Email`** — a more reliable unique identifier than name, since two different people can share a name but not a real email address.

In [13]:
exact_dupes = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {exact_dupes} exact duplicate rows")

Removed 3 exact duplicate rows


In [14]:
df['Email_clean'] = df['Email'].str.strip().str.lower()

before = len(df)
df = df.drop_duplicates(subset=['Email_clean'], keep='first')
near_dupes_removed = before - len(df)
print(f"Removed {near_dupes_removed} near-duplicate rows (same email, keeping the first occurrence)")

df = df.drop(columns=['Email_clean'])

Removed 2 near-duplicate rows (same email, keeping the first occurrence)


## Step 6: Standardize Text Fields

Next I made sure the same real-world value is always represented the same way:
- Stripped leading/trailing whitespace from names, emails, and universities
- Title-cased names, lower-cased emails (the standard convention)
- Mapped every spelling/casing variant of `Domain Applied` to one of the 5 correct labels using a dictionary
- Title-cased `Status` so `"selected"`, `"Selected"`, and `"SELECTED"` all become one value

After mapping the domain column, I re-ran `.unique()` to confirm exactly 5 clean values remain, and that no unmapped variation slipped through as `NaN`.

In [15]:
df['Applicant Name'] = df['Applicant Name'].str.strip().str.title()
df['Email'] = df['Email'].str.strip().str.lower()
df['University'] = df['University'].str.strip()

In [16]:
domain_mapping = {
    'web dev': 'Web Development', 'Web Development': 'Web Development', 'WEB DEV': 'Web Development',
    'webdev': 'Web Development', 'Web Dev': 'Web Development',
    'app dev': 'App Development', 'App Development': 'App Development', 'APP DEV': 'App Development',
    'appdev': 'App Development',
    'data science': 'Data Science', 'Data Science': 'Data Science', 'DATA SCIENCE': 'Data Science',
    'Data-Science': 'Data Science',
    'graphic design': 'Graphic Design', 'Graphic Design': 'Graphic Design', 'GRAPHIC DESIGN': 'Graphic Design',
    'digital marketing': 'Digital Marketing', 'Digital Marketing': 'Digital Marketing',
    'DIGITAL MKTG': 'Digital Marketing',
}

df['Domain Applied'] = df['Domain Applied'].str.strip().map(domain_mapping)


print(df['Domain Applied'].isnull().sum(), "unmapped values")
df['Domain Applied'].unique()

0 unmapped values


array(['Graphic Design', 'App Development', 'Digital Marketing',
       'Web Development', 'Data Science'], dtype=object)

In [17]:
df['Status'] = df['Status'].str.strip().str.title()
df['Status'].unique()

array(['Under Review', 'Rejected', 'Selected'], dtype=object)

## Step 7: Fix Data Types (Dates and Phone Numbers)

`Application Date` looks fine to a human but is stored as plain text, so it can't be sorted or filtered as a real date. I converted it with `pd.to_datetime`, using:
- `format='mixed'` so pandas can correctly parse the several different formats present in the raw data instead of forcing one format on every row
- `errors='coerce'` so any truly unreadable value (like the placeholder `"not_provided"`) becomes `NaT` instead of crashing the whole notebook

I then re-checked how many nulls appeared *after* conversion those are the rows where the original date genuinely couldn't be understood, not a sign of a mistake.

I also cleaned `Phone` by removing dashes and spaces, but kept it as text converting it to a number would drop the leading zero that Pakistani mobile numbers start with.

In [18]:
nulls_before = df['Application Date'].isnull().sum()

df['Application Date'] = pd.to_datetime(
    df['Application Date'], errors='coerce', format='mixed', dayfirst=True
)

nulls_after = df['Application Date'].isnull().sum()
print(f"{nulls_after - nulls_before} dates could not be parsed and became NaT")

3 dates could not be parsed and became NaT


In [19]:
df['Phone'] = (
    df['Phone'].astype(str)
    .str.replace('-', '', regex=False)
    .str.replace(' ', '', regex=False)
)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 84 entries, 0 to 94
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    84 non-null     object        
 1   Email             84 non-null     object        
 2   Phone             84 non-null     object        
 3   Domain Applied    84 non-null     object        
 4   University        84 non-null     object        
 5   Application Date  81 non-null     datetime64[ns]
 6   Status            84 non-null     object        
dtypes: datetime64[ns](1), object(6)
memory usage: 5.2+ KB


`Application Date` is now confirmed as `datetime64` type instead of `object` (text).

## Step 8: Data Quality Summary

**Starting dataset:** 95 rows, 7 columns

**Issues found:**
- Missing values: `Applicant Name` (3), `Email` (5), `University` (15), `Status` (9)
- 3 exact duplicate rows, plus 2 additional duplicate applications identified by matching `Email`
- `Domain Applied` had multiple spelling/casing variations for only 5 real domains
- `Application Date` was stored as text, in four different formats, including some unreadable placeholder values
- `Phone` numbers contained inconsistent dash formatting

**Actions taken:**
- Converted blank strings to real `NaN` so missing values could actually be detected
- Dropped rows with missing `Applicant Name` (unusable without identification)
- Dropped rows with missing `Email` (needed for follow-up and used as the deduplication key)
- Filled missing `University` with `"Not Specified"` (non-critical field)
- Filled missing `Status` with `"Under Review"` (sensible default for unprocessed applications)
- Removed exact duplicate rows, then removed near-duplicates by matching cleaned `Email`
- Standardized all `Domain Applied` values into exactly 5 consistent categories
- Standardized `Status` casing into exactly 3 consistent categories
- Converted `Application Date` to a proper `datetime64` type, coercing unreadable dates to `NaT`
- Cleaned `Phone` number formatting for consistency (kept as text to preserve leading zeros)

**Final dataset:** cleaned, no missing values in critical fields, no duplicates, consistent formatting throughout. See the exact final row/column count and remaining `NaT` date count printed below.

In [20]:
print("Final shape:", df.shape)
print("\nRemaining nulls by column:")
print(df.isnull().sum())
print("\nRemaining duplicate rows:", df.duplicated().sum())

Final shape: (84, 7)

Remaining nulls by column:
Applicant Name      0
Email               0
Phone               0
Domain Applied      0
University          0
Application Date    3
Status              0
dtype: int64

Remaining duplicate rows: 0


## Step 9: Export the Cleaned Dataset

`index=False` is important here — without it, Pandas would add an extra unnamed column of row numbers to the CSV that nobody needs.

In [21]:
df.to_csv('applicants_cleaned.csv', index=False)
print("Saved applicants_cleaned.csv")

Saved applicants_cleaned.csv


In [22]:
# Final sanity check: reload the exported file and confirm it looks right
check = pd.read_csv('applicants_cleaned.csv')
check.head()

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,David Jenkins,georgemiller@example.com,3937923747,Graphic Design,Not Specified,2025-10-16,Under Review
1,Martin Ross,sheila14@example.org,3940196556,Graphic Design,LUMS,2026-02-21,Rejected
2,Dana Terry,melissa91@example.net,3058651850,App Development,Comsats University,2026-05-21,Under Review
3,Carmen Rose,ibrandt@example.net,3346578713,Digital Marketing,Punjab University,2025-05-10,Under Review
4,Eric Ortiz,jterry@example.org,3366299468,Web Development,Punjab University,NaN,Under Review
